# CSE 164 Shared ConvNeXt Multi-Task Pipeline\n
\n
This notebook runs the from-scratch shared ConvNeXt multi-task pipeline in Google Colab or on a Google Cloud VM.\n
\n
Expected Drive layout:\n
\n
```text\n
/content/drive/MyDrive/CSE164FinalProject/raw/\n
|-- metadata/\n
|-- test/\n
|-- train_labeled/\n
|-- train_seg/\n
|-- train_unlabeled/\n
`-- val/\n
```\n
\n
All model outputs are written to Drive under `/content/drive/MyDrive/CSE164FinalProject/outputs/`, so checkpoints, figures, and `submission.csv` survive the Colab runtime.

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Unzip the datset

In [17]:
!mkdir -p /content/data

!cp /content/drive/MyDrive/CSE164FinalProject/raw.zip /content/raw.zip

!unzip -q /content/raw.zip -d /content/data

!ls /content/data
!ls /content/data/raw

raw
metadata  test	train_labeled  train_seg  train_unlabeled  val


## Configuration\n
\n
Change these values before running if your Drive folder or training settings are different.

In [18]:
from pathlib import Path

GITHUB_REPO = 'https://github.com/007deepaks/CSE-164FinalProject.git'
REPO_DIR = Path('/content/CSE-164FinalProject')

DRIVE_PROJECT = Path('/content/drive/MyDrive/CSE164FinalProject')
DATA_ROOT = Path('/content/data/raw')
OUTPUT_ROOT = DRIVE_PROJECT / 'outputs'

CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
SUBMISSION_DIR = OUTPUT_ROOT / 'submissions'
PREDICTION_DIR = OUTPUT_ROOT / 'predictions'

EPOCHS = 40
SEG_BATCH_SIZE = 2
CLS_BATCH_SIZE = 16
VAL_BATCH_SIZE = 2
MODEL_SIZE = 'small'
IMAGE_SIZE = 320
NUM_WORKERS = 2

# Set these for a quick Colab sanity check before full training.
MAX_SEG_SAMPLES = None
MAX_CLS_SAMPLES = None
MAX_VAL_SAMPLES = None

for path in [CHECKPOINT_DIR, FIGURE_DIR, SUBMISSION_DIR, PREDICTION_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Data root:', DATA_ROOT)
print('Outputs:', OUTPUT_ROOT)

Data root: /content/data/raw
Outputs: /content/drive/MyDrive/CSE164FinalProject/outputs


## Clone or Update Repository\n
\n
If the repo is private and this cell fails, make the repo public temporarily or clone with a GitHub token/Colab GitHub auth. Do not hard-code a token into this notebook before committing.

In [19]:
import getpass

token = getpass.getpass("GitHub Token: ")

if REPO_DIR.exists():
    %cd {REPO_DIR}
    !git pull
else:
    %cd /content
    !git clone https://{token}@github.com/007deepaks/CSE-164FinalProject.git
    %cd {REPO_DIR}

!git status --short

GitHub Token: ··········
/content/CSE-164FinalProject
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 5.04 KiB | 1.26 MiB/s, done.
From https://github.com/007deepaks/CSE-164FinalProject
   99a7b3d..9cdb437  main       -> origin/main
Updating 99a7b3d..9cdb437
Fast-forward
 notebooks/train_colab.ipynb | 832 +++++++++++++++++++++++++++-----------------
 1 file changed, 504 insertions(+), 328 deletions(-)


## Install Light Dependencies\n
\n
Colab normally includes CUDA-enabled PyTorch already, so this avoids reinstalling `torch` from `requirements.txt` and only ensures the lightweight packages are available.

In [20]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

!python -m pip install -q numpy pandas pillow

torch: 2.11.0+cu128
cuda available: True
gpu: Tesla T4


## Verify Dataset Layout\n
\n
This should show `metadata`, `test`, `train_labeled`, `train_seg`, `train_unlabeled`, and `val` inside `raw/`.

In [21]:
assert DATA_ROOT.exists(), f'Missing DATA_ROOT: {DATA_ROOT}'

!find {DATA_ROOT} -maxdepth 2 -type d | sort | head -40

!python -m src.data.inspect_dataset \
    --data-root {DATA_ROOT} \
    --sample-masks 3 \
    --frequency-masks 25 \
    --random-seed 164 \
    --pair-check-limit 100

/content/data/raw
/content/data/raw/metadata
/content/data/raw/test
/content/data/raw/test/images
/content/data/raw/train_labeled
/content/data/raw/train_labeled/images
/content/data/raw/train_seg
/content/data/raw/train_seg/images
/content/data/raw/train_seg/masks
/content/data/raw/train_unlabeled
/content/data/raw/train_unlabeled/images
/content/data/raw/val
/content/data/raw/val/images
/content/data/raw/val/masks
Dataset root: /content/data/raw

Metadata
  class_map.json: list, entries=300
    first: {'class_id': 0, 'name': 'class_000', 'segmentation_id': 1}
  train_labeled.json: list, entries=7500
    first: {'class_id': 0, 'image': 'train_labeled/images/labeled_000_000.JPEG'}
  train_seg.json: list, entries=3000
    first: {'class_id': 0, 'image': 'train_seg/images/seg_000_000.JPEG', 'mask': 'train_seg/masks/seg_000_000.png', 'segmentation_id': 1}
  val/classification.json: entries=750

Split counts and sampled image sizes
  train_labeled
    images: 7500 files
    sampled sizes: 

## Train Shared Multi-Task Model\n
\n
For a quick sanity run, set `MAX_SEG_SAMPLES = 4`, `MAX_CLS_SAMPLES = 8`, `MAX_VAL_SAMPLES = 8`, `SEG_BATCH_SIZE = 1`, `CLS_BATCH_SIZE = 2`, and `EPOCHS = 1` in the config cell.

In [24]:
!python -m src.training.train_multitask \
  --data-root "$DATA_ROOT" \
  --epochs "$EPOCHS" \
  --model-size "$MODEL_SIZE" \
  --seg-batch-size "$SEG_BATCH_SIZE" \
  --cls-batch-size "$CLS_BATCH_SIZE" \
  --val-batch-size "$VAL_BATCH_SIZE" \
  --image-size "$IMAGE_SIZE" \
  --num-workers "$NUM_WORKERS" \
  --checkpoint-dir "$CHECKPOINT_DIR"

Using device: cuda; mixed precision: True
Train samples: 3000; val samples: 750
Scheduler: CosineAnnealingLR T_max=10, eta_min=3e-05

Epoch 1/10
  train batch 0020/375 loss=4.7264
  train batch 0040/375 loss=4.2598
  train batch 0060/375 loss=4.0285
  train batch 0080/375 loss=3.9803
  train batch 0100/375 loss=3.8955
  train batch 0120/375 loss=3.8457
  train batch 0140/375 loss=3.7197
  train batch 0160/375 loss=3.5563
  train batch 0180/375 loss=3.1984
  train batch 0200/375 loss=3.6149
  train batch 0220/375 loss=2.6262
  train batch 0240/375 loss=3.5536
  train batch 0260/375 loss=2.8718
  train batch 0280/375 loss=2.7752
  train batch 0300/375 loss=2.7006
  train batch 0320/375 loss=2.8781
  train batch 0340/375 loss=1.8664
  train batch 0360/375 loss=2.9735
  train batch 0375/375 loss=3.4496
/content/CSE-164FinalProject/src/training/train_segmentation.py:52: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray

## Evaluate Best Checkpoint\n
\n
This reports the Kaggle-like validation metrics for the best multi-task checkpoint.

In [27]:
BEST_CHECKPOINT = CHECKPOINT_DIR / 'best_multitask.pt'
assert BEST_CHECKPOINT.exists(), f'Missing checkpoint: {BEST_CHECKPOINT}'

!python -m src.training.evaluate_multitask \
  --checkpoint "$BEST_CHECKPOINT" \
  --data-root "$DATA_ROOT" \
  --image-size "$IMAGE_SIZE" \
  --batch-size "$VAL_BATCH_SIZE" \
  --num-workers 0

{
  "loss": 2.430451396931993,
  "mean_iou": 0.0,
  "pixel_accuracy": 0.7058326883026427
}


## Visualize Validation Predictions\n
\n
This writes labeled validation panels with image, ground truth, prediction, overlay, and difference views.

In [28]:
!python -m src.visualization.visualize_val_predictions \
  --checkpoint "$BEST_CHECKPOINT" \
  --data-root "$DATA_ROOT" \
  --split val \
  --image-size "$IMAGE_SIZE" \
  --num-samples 12 \
  --output-dir "$FIGURE_DIR"

  predicted batch 0020/3000
  predicted batch 0040/3000
  predicted batch 0060/3000
  predicted batch 0080/3000
  predicted batch 0100/3000
  predicted batch 0120/3000
  predicted batch 0140/3000
  predicted batch 0160/3000
  predicted batch 0180/3000
  predicted batch 0200/3000
  predicted batch 0220/3000
  predicted batch 0240/3000
  predicted batch 0260/3000
  predicted batch 0280/3000
  predicted batch 0300/3000
  predicted batch 0320/3000
  predicted batch 0340/3000
  predicted batch 0360/3000
  predicted batch 0380/3000
  predicted batch 0400/3000
  predicted batch 0420/3000
  predicted batch 0440/3000
  predicted batch 0460/3000
  predicted batch 0480/3000
  predicted batch 0500/3000
  predicted batch 0520/3000
  predicted batch 0540/3000
  predicted batch 0560/3000
  predicted batch 0580/3000
  predicted batch 0600/3000
  predicted batch 0620/3000
  predicted batch 0640/3000
  predicted batch 0660/3000
  predicted batch 0680/3000
  predicted batch 0700/3000
  predicted batch 07

## Generate Kaggle Submission\n
\n
The final CSV is written to Drive at `CSE164FinalProject/outputs/submissions/submission.csv`.

In [ ]:
SUBMISSION_CSV = SUBMISSION_DIR / 'submission.csv'

!python -m src.training.predict_multitask \
  --checkpoint "$BEST_CHECKPOINT" \
  --data-root "$DATA_ROOT" \
  --output "$SUBMISSION_CSV" \
  --image-size "$IMAGE_SIZE" \
  --batch-size "$VAL_BATCH_SIZE" \
  --num-workers 0

print('Submission stored in Drive at:', SUBMISSION_CSV)

## Inspect Output Files

In [29]:
import pandas as pd

print('Checkpoints:')
for path in sorted(CHECKPOINT_DIR.glob('*')):
    print(path.name, f'{path.stat().st_size / (1024 * 1024):.1f} MB')

print('\nRecent figures:')
for path in sorted(FIGURE_DIR.glob('*'))[-10:]:
    print(path.name)

print('\nSubmissions:')
for path in sorted(SUBMISSION_DIR.glob('*.csv')):
    print(path.name, f'{path.stat().st_size / (1024 * 1024):.1f} MB')

df = pd.read_csv(SUBMISSION_CSV)

print('\nSubmission preview:')
print(df.head())

print('rows:', len(df))
print('columns:', list(df.columns))

Checkpoints:
best_segmentation.pt 38.3 MB
latest_segmentation.pt 38.3 MB
segmentation_history.json 0.0 MB

Recent figures:
epoch_010_val_000.jpg
epoch_010_val_001.jpg
epoch_010_val_002.jpg
epoch_010_val_003.jpg
eval_val_000.jpg
eval_val_001.jpg
eval_val_002.jpg
eval_val_003.jpg
eval_val_004.jpg
eval_val_005.jpg

Submissions:
submission.csv 1.2 MB

Submission preview:
             image  class_id  \
0  test_00000.JPEG         0   
1  test_00001.JPEG         0   
2  test_00002.JPEG         0   
3  test_00003.JPEG         0   
4  test_00004.JPEG       206   

                                    segmentation_rle  
0                                                  0  
1                                                  0  
2                                                  0  
3                                                  0  
4  101138 2 207 101141 3 207 101146 1 207 101637 ...  
rows: 3000
columns: ['image', 'class_id', 'segmentation_rle']
